# RoadVision — Colab / Kaggle runner
Runs the full pipeline on a free T4/P100 GPU. Runtime → Change runtime type → GPU.

Total time on a T4 with default config (≈ 30k images, Norway excluded): roughly 8–12 h for tuning + training. For a faster first pass set `optuna.yolo_trials: 8`, `yolo.epochs: 60`.

In [ ]:
!nvidia-smi
# 1) get the code (upload the zip or clone your GitHub repo)
# !git clone https://github.com/<you>/RoadVision.git
%cd RoadVision
!pip -q install -r requirements.txt

In [ ]:
# 2) data (≈2.5 GB without Norway)
!python scripts/run_pipeline.py --stages download prepare

In [ ]:
# 3) Bayesian HPO (Optuna + MLflow)  — optional but scored in the rubric
!python scripts/run_pipeline.py --stages tune_yolo

In [ ]:
# 4) final training: YOLO -> crops (hard-negative mining) -> ViT tuning + training -> RT-DETR
!python scripts/run_pipeline.py --stages train_yolo crops tune_vit train_vit train_rtdetr

In [ ]:
# 5) calibration on val, test evaluation, benchmark, explainability, all figures, LaTeX tables
!python scripts/run_pipeline.py --stages evaluate benchmark explain plots report

In [ ]:
# 6) (optional) StripRFNet-comparable protocol on the China subset
!python scripts/run_pipeline.py --override configs/china_811.yaml --stages prepare train_yolo crops train_vit train_rtdetr evaluate benchmark

In [ ]:
import json; r=json.load(open('runs/metrics/eval_results.json'))
{k:(round(v['f1'],3), round(v['mAP50'],3), round(v['fps'],1)) for k,v in r['modes'].items()}

In [ ]:
# 7) save weights + figures + tables to your Drive / download
!zip -r roadvision_outputs.zip models runs/metrics runs/figures report/tables report/figures mlruns